In [1]:
# Enable autoreload and black formatting in Jupyter notebooks
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

In [2]:
import os
import re
import polars as pl
import pandas as pd
import duckdb as dd

In [3]:
# Get current working directory
cwd = os.getcwd()
# Construct the path to the data directory
data_dir = os.path.normpath(
    os.path.join(cwd, "..", "..", "Data storage Siemens", "Playground")
)

# Check if the directory exists
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"The directory {data_dir} does not exist.")
else:
    print(f"Data storage directory found: {data_dir}")

Data storage directory found: /mnt/ceph/vol_02_home_students/studraabf1/PhD/MRI_data_analysis/Data storage Siemens/Playground


## Prepare Merging Process

In [ ]:
def set_up_duckdb(cwd: str) -> dd.DuckDBPyConnection:
    """
    Set up a DuckDB connection to a database file in the specified data directory.

    Parameters:
        cwd (str): The current working directory.

    Returns:
        dd.DuckDBPyConnection: A connection object to the DuckDB database.
    """

    # Create a duckdb folder if it doesn't exist
    duckdb_dir = os.path.join(cwd, "duckdb_files")
    os.makedirs(duckdb_dir, exist_ok=True)
    print(f"DuckDB files directory is at: {duckdb_dir}")

    # Connect to a DuckDB database file in the duckdb_files directory
    duckdb_path = os.path.join(duckdb_dir, "MRI_data.duckdb")
    con = dd.connect(
        duckdb_path, read_only=False
    )  # Ensure read_only=False for write access
    print(f"DuckDB database created at: {duckdb_path}")

    # Set the temporary directory for DuckDB to the duckdb_files directory
    con.execute(f"SET temp_directory='{duckdb_dir}'")
    print(f"Temporary directory for DuckDB set to: {duckdb_dir}")

    return con


con = set_up_duckdb(cwd)

DuckDB files directory is at: /mnt/ceph/vol_02_home_students/studraabf1/PhD/MRI_data_analysis/Data preprocessing + pre-analysis/Data preprocessing_opt/duckdb_files
DuckDB database created at: /mnt/ceph/vol_02_home_students/studraabf1/PhD/MRI_data_analysis/Data preprocessing + pre-analysis/Data preprocessing_opt/MRI_data.duckdb


In [5]:
def get_mri_folders(data_dir: str, dir_exclude: list = []) -> list:
    """
    The data of the MRI scanners is stored in dedictaed folders for each MRI
    scanner.  For further processing get a list of all MRI folder. Always
    exclude the DuckDB folder and CSV files which were created to allow the merging.
    Additionally, furtherdirectories can be excluded.


    Parameters:
        data_dir (str): Path to the data directory.
        dir_exclude (list): List of directory names to exclude.

    Returns:
        list: List of MRI folder names.
    """

    # Get the list of all folders which should correspond to MRI scanners.
    # Exclude the folder that contains the DuckDB temp files and any CSV files which
    # were created to allow the merging
    return [
        folder
        for folder in os.listdir(data_dir)
        if (
            not folder.startswith("duckdb")
            and not folder.endswith(".csv")
            and folder not in dir_exclude
        )
    ]


mri_folders = get_mri_folders(data_dir)
print(mri_folders)

['ukt_69667', 'ukt_75609']


In [6]:
def check_mri_folder(data_dir: str, mri_folders: list) -> bool:
    """
    Control if the data directory contains folders corresponding to MRI scanners by
    checking if the given Scanner.csv file contains a 'MR' pattern in the
    SiteSecondaryName column which indicates that the parent folder corresponds
    to an MRI scanner.

    Parameters:
        data_dir (str): Path to the data directory.
        mri_folders (list): List of MRI folder names.
    Returns:
        bool: True if the 'MR' pattern is found, False otherwise.
    """

    # Iterate over MRI folders
    for mri_folder in mri_folders:

        # Construct the path to the MRI folder
        mri_folder_path = os.path.join(data_dir, mri_folder)

        # Get the distinct SiteSecondaryName values from all Scanner.csv files in
        # the MRI folder
        scanner_df = con.execute(
            f"""
            SELECT DISTINCT
                SiteSecondaryName
            FROM 
                read_csv_auto('{mri_folder_path}/*/Scanner.csv', union_by_name=true)
            WHERE 
                SiteSecondaryName IS NOT NULL
            """
        ).df()

        # Check if any of the SiteSecondaryName values contain the pattern 'MR'
        # which indicates that the parent folder corresponds to an MRI scanner
        mri_found = (
            scanner_df["SiteSecondaryName"].astype(str).str.contains(r"MR\s*\d*").any()
        )

        # Raise an error if no matching 'MR' pattern is found for the current
        # MRI folder
        if not mri_found:
            raise Warning(
                f"""
                Folder {mri_folder} does not contain the specification of a 'MR' 
                pattern in the SiteSecondaryName column of any Scanner.csv file.
                Please ensure that the parent folders correspond to an MRI scanner.
                """
            )

    print("No invalid MRI folders found. Continuing with the script...")


check_mri_folder(data_dir, mri_folders)

No invalid MRI folders found. Continuing with the script...


In [7]:
def create_mapping_scanner_mapping(data_dir: str, mri_folders: list) -> pd.DataFrame:
    """
    Create a mapping between the ScannerID and the corresponding powermeter data
    files by iterating over the MRI folders and reading the Scanner.csv files to
    extract the ScannerID and by extracting all csv files name which start with
    powerdata_*.csv. This has to be done manually since no csv file exists which
    contains this mapping. It is assumed that there is only one unique ScannerID
    per MRI folder. On the contrary, there can be multiple powerdata_*.csv files
    for each MRI folder which should all be included in the mapping.

    Parameters:
        data_dir (str): Path to the data directory.
        mri_folders (list): List of MRI folder names.

    Returns:
        pd.DataFrame: DataFrame containing the mapping between ScannerID and
                      powermeter data files.
        - Creates also a CSV file in the data directory with the mapping.
    """
    # Initialize an empty list to store the mappings
    mappings = []

    # Iterate over MRI folders
    for mri_folder in mri_folders:
        # Get unique ScannerID assuming one per folder
        scanner_df = con.execute(
            f"""
            SELECT DISTINCT 
                ScannerID,
                Serial
            FROM 
                read_csv_auto('{data_dir}/{mri_folder}/*/Scanner.csv', union_by_name=true)
            """
        ).df()

        # Skip if no scanner data
        if scanner_df.empty:
            continue

        # Get unique ScannerIDs, which should be one per folder
        scanner_id = scanner_df["ScannerID"].unique()
        serial = scanner_df["Serial"].unique()

        # If there are multiple unique ScannerIDs, raise an error since we expect only
        # one ScannerID per MRI folder
        if len(scanner_id) > 1 or len(serial) > 1:
            raise ValueError(
                f"Multiple unique ScannerIDs and Serials found in folder {mri_folder}: {scanner_id}, {serial}"
            )

        # Get unique Powermeter_IDs which can be multiple per folder
        power_df = con.execute(
            f"""
            SELECT DISTINCT 
                filename
            FROM 
                read_csv_auto('{data_dir}/{mri_folder}/*/powerdata_*.csv', filename=true, union_by_name=true)
            """
        ).df()

        # Skip if no power data
        if power_df.empty:
            continue

        # Extract just the powerdata_*.csv part from the filename to get Powermeter_ID
        power_df["PowermeterID"] = power_df["filename"].apply(
            lambda x: os.path.splitext(os.path.basename(x))[0].split("_", 1)[1]
        )

        # Get unique power IDs
        power_ids = power_df["PowermeterID"].unique()

        # Create mappings
        for power_id in power_ids:
            mappings.append(
                {
                    "ScannerID": scanner_id[0],
                    "Serial": serial[0],
                    "PowermeterID": power_id,
                }
            )

    # Create DataFrame and save to CSV
    powermeter_scanner_mapping_df = pd.DataFrame(mappings).drop_duplicates()
    output_path = os.path.join(data_dir, "powermeter_scanner_mapping.csv")
    powermeter_scanner_mapping_df.to_csv(output_path, index=False, sep=";")

    # Sort by Scanner_ID and Powermeter_ID for better readability
    powermeter_scanner_mapping_df = powermeter_scanner_mapping_df.sort_values(
        by=["ScannerID", "Serial", "PowermeterID"]
    ).reset_index(drop=True)

    # Register the DataFrame as a DuckDB table for further use in SQL queries
    con.register("powermeter_scanner_mapping", powermeter_scanner_mapping_df)

    return powermeter_scanner_mapping_df


powermeter_scanner_mapping_df = create_mapping_scanner_mapping(data_dir, mri_folders)
display(powermeter_scanner_mapping_df)

,ScannerID,Serial,PowermeterID
0,52,69667,LQN230413610168
1,55,75609,LQN230413610160
2,55,75609,LQN230413610162


## Merging process

## Parameters

In [8]:
def get_parameters_description(data_dir: str) -> pd.DataFrame:
    """
    Get the ProtocolParametersDescription.csv files from all MRI folders, join them
    into a single DataFrame, and drop duplicates. This ensures that we have a
    comprehensive and unique set of protocol parameter descriptions across all MRI scanners.

    Parameters:
        data_dir (str): Path to the data directory.

    Returns:
        tuple: A tuple containing:
            - pd.DataFrame: DataFrame containing the joined ProtocolParametersDescription data
                            with duplicates dropped.
            - dict: A dictionary for renaming columns, mapping ProtocolParameterName to Explanation.
    """

    # Get all joined ProtocolParametersDescription.csv files into a single DataFrame
    joined_protcol_parameters_df = con.execute(
        f"""
        SELECT 
            *
        FROM 
            read_csv_auto('{data_dir}/*/*/ProtocolParametersDescription.csv', union_by_name=true)
        AS 
            protcol_params   
        WHERE 
            protcol_params.Explanation IS NOT NULL AND
            protcol_params.ProtocolParameterName IS NOT NULL
        ORDER BY
            protcol_params.ProtocolParameterName
        """
    ).df()

    # Drop duplicates
    param_renaming_df = joined_protcol_parameters_df.drop_duplicates()

    # Sort by the ProtocolParameterName for better readability
    param_renaming_df = param_renaming_df.sort_values(
        by="ProtocolParameterName"
    ).reset_index(drop=True)

    # Define renaming dictionary by zipping the ProtocolParameterName and Explanation
    # columns from the unique protocol parameters DataFrame
    param_renaming_dict = dict(
        zip(
            param_renaming_df.ProtocolParameterName,
            param_renaming_df.Explanation,
        )
    )

    return param_renaming_df, param_renaming_dict


param_renaming_df, param_renaming_dict = get_parameters_description(data_dir)
display(param_renaming_df)

,ProtocolParameterName,Explanation
0,AAM,AutoAlignRefMode
1,AAR,AutoAlignRegion
2,ACC,ACC or SOS
3,ACQW,acquisition window
4,ADI,adiabat mode
...,...,...
182,VER,version number
183,VISH,view sharing
184,WARPS,WARP SEMAC steps
185,WARPV,WARP VAT


In [9]:
def get_rename_parameter(data_dir: str, renaming_dict: dict) -> pd.DataFrame:
    """
    Get the ProtocolParameters.csv files from all MRI folders and rename the
    columns using the provided renaming dictionary. This allows us to have more
    descriptive column names based on the explanations provided in the
    ProtocolParametersDescription.csv files.

    Parameters:
        data_dir (str): Path to the data directory.
        renaming_dict (dict): A dictionary for renaming columns, mapping
                                ProtocolParameterName to Explanation.

    Returns:
        pd.DataFrame: A DataFrame containing the ProtocolParameters data with
                                renamed columns.

    """

    # Read in the ProtocolParameters.csv files
    params_df = con.execute(
        f"""
        SELECT
            *
        FROM
            read_csv_auto('{data_dir}/*/*/ProtocolParameters.csv', union_by_name=true) AS protocol_params
        """
    ).df()

    # Rename the columns of the parameters DataFrame using the renaming dictionary
    renamed_params_df = params_df.rename(columns=renaming_dict, inplace=False)

    # Check if the renaming was successful by comparing the columns before and after renaming
    # Columns which should be renamed are those that are present in both the
    # renaming dictionary and the parameters DataFrame
    to_rename_cols = set(renaming_dict.keys()) & set(params_df.columns)
    # The expected new column names after renaming
    expected_new_cols = {renaming_dict[k] for k in to_rename_cols}
    # The renamed columns in the DataFrame
    renamed_cols = set(renamed_params_df.columns)
    # Columns that should have been created by renaming but are missing
    missing_renamed = expected_new_cols - renamed_cols

    # Columns that should have been renamed but are still present
    if missing_renamed:
        raise Warning(
            f"""
            Renaming issue detected:
            Columns that should have been created by renaming but are missing: {missing_renamed}
            Please check the renaming dictionary and ensure all ProtocolParameterName values have a corresponding Explanation.
            """
        )

    # Add a suffix to the column names "_params"
    renamed_params_df = renamed_params_df.add_suffix("_params")

    # Register the renamed DataFrame as a DuckDB table
    con.register("renamed_params", renamed_params_df)

    return renamed_params_df


renamed_params_df = get_rename_parameter(data_dir, param_renaming_dict)
display(renamed_params_df)

,FK_MeasurementID_params,TR_params,SliceOversampling_params,Flip Angle (array1)_params,Concats_params,flow compensation_params,Reduced EC sensit._params,reordering_params,RF pulse type_params,AutoAlignRegion_params,...,FatSat_params,view sharing_params,Image filter_params,slew rate fast*_params,PositioningMode_params,Dixon Fast_params,2nd TI time_params,Readout segments_params,RPF_params,Local Shim_params
0,1315673197803,7330,<NA>,<NA>,1,2,False,<NA>,2,1,...,<NA>,<NA>,1,2,1,None,<NA>,<NA>,<NA>,<NA>
1,1316211424275,2500,10,<NA>,1,1,False,1,1,1,...,0,1,<NA>,1,1,None,<NA>,<NA>,<NA>,<NA>
2,1316211443652,3000,<NA>,<NA>,1,4,False,<NA>,1,1,...,<NA>,<NA>,1,1,1,None,<NA>,<NA>,<NA>,<NA>
3,1315673199370,5500,<NA>,<NA>,1,2,False,<NA>,2,1,...,<NA>,<NA>,1,2,1,None,<NA>,<NA>,<NA>,<NA>
4,1316211450256,450,<NA>,<NA>,<NA>,2,False,<NA>,2,1,...,<NA>,<NA>,<NA>,1,1,None,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1344,1470079439259,2330,<NA>,90,2,0,False,<NA>,2,<NA>,...,<NA>,<NA>,1,<NA>,<NA>,None,<NA>,<NA>,<NA>,4
1345,1470079439374,744,<NA>,<NA>,1,<NA>,False,<NA>,2,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,None,<NA>,<NA>,<NA>,1
1346,1470079439457,5970,<NA>,90,1,0,True,<NA>,2,<NA>,...,<NA>,<NA>,1,<NA>,<NA>,None,<NA>,<NA>,<NA>,2
1347,1470079439561,2210,<NA>,90,2,0,True,<NA>,2,<NA>,...,<NA>,<NA>,1,<NA>,<NA>,None,<NA>,<NA>,<NA>,4


## Measurements

In [10]:
def get_measurement(data_dir: str) -> pd.DataFrame:
    """
    Get the Measurements.csv files from all MRI folders, join them into a single
    DataFrame, and perform cleaning steps to create a unified energy measurement
    column. This includes identifying columns that start with "TotalEnergy", summing
    them to create a new "SiemensTotalEnergy" column, and replacing 0 values with NaN.
    This is necessary because the energy measurements are stored in different columns for
    different MRI scanners, but they all start with "TotalEnergy" which allows us to
    identify and unify them.

    Parameters:
        data_dir (str): Path to the data directory.
    Returns:
        pd.DataFrame: A cleaned DataFrame containing the measurements data with a unified energy column.
    """

    measurements_df = con.execute(
        f"""
    SELECT 
        *
    FROM 
        read_csv_auto('{data_dir}/*/*/Measurements.csv', union_by_name=true) AS measurements
    """
    ).df()

    # Get the columns that start with "TotalEnergy" which indicate the energy
    # measurements. For each MRI scanner the energy column is differently named
    # but they all start with "TotalEnergy" which allows us to identify them
    energy_columns = [
        col for col in measurements_df.columns if col.startswith("TotalEnergy")
    ]

    # Get all the rows where the TotalEnergy columns are all NaN to check if there
    # are measurements without energy data
    measurements_without_energy = measurements_df[energy_columns].isna().all(axis=1)

    # Add all columns starting with "TotalEnergy" to create a new column "IntermediateEnergy".
    # By default, pandas sum ignores NaN values and sums the available values.
    # However, if all values in a row are NaN, the result will be 0.
    # To ensure the result is NaN when all values are NaN,
    # we identify such rows beforehandand set the sum to NaN for them afterwards.
    measurements_df["SiemensTotalEnergy_KWh"] = measurements_df[energy_columns].sum(
        axis=1, skipna=True
    )

    # Drop all columns starting with "TotalEnergy"
    measurements_df = measurements_df.drop(columns=energy_columns, inplace=False)

    # Set the measurements without energy data to NaN in the SiemensTotalEnergy_KWh column
    measurements_df.loc[measurements_without_energy, "SiemensTotalEnergy_KWh"] = pd.NA

    # Add a suffix to the column names "_meas"
    measurements_df = measurements_df.add_suffix("_meas")

    # Register the cleaned measurements DataFrame as a DuckDB table
    con.register("measurements", measurements_df)

    return measurements_df


measurements_df = get_measurement(data_dir)
display(measurements_df)

,MeasurementID_meas,MeasurementStart_meas,MeasurementEnd_meas,SeqNumMeasPerExam_meas,ScanningTime_meas,NoScanningTime_meas,MeasurementType_meas,BodyRegion_meas,Sequence_meas,Protocol_meas,...,HasMeasFailOrStop_meas,SARPrepDuration_meas,SARManualEventCount_meas,AdjustPrepFailsCount_meas,MeasPrepDuration_meas,MeasStopPenaltyDuration_meas,PrepFailOrStopCount_meas,MeasFailOrStopCount_meas,SqueezeBallCount_meas,SiemensTotalEnergy_KWh_meas
0,1315673187032,2024-06-05 07:15:56,2024-06-05 07:15:57,<NA>,1,1,None,None,%AdjustSeq%/AdjDicoSeq,AdjDico,...,False,None,None,None,<NA>,False,False,False,False,NaN
1,1315673187047,2024-06-05 07:15:58,2024-06-05 07:15:58,<NA>,0,1,None,None,%AdjustSeq%/AdjDicoSeq,AdjDico,...,False,None,None,None,<NA>,False,False,False,False,NaN
2,1315673187066,2024-06-05 07:16:00,2024-06-05 07:16:00,<NA>,0,0,None,None,%AdjustSeq%\STIMOFunctionTest,Initialized by sequence,...,False,None,None,None,<NA>,False,False,False,False,NaN
3,1315673187074,2024-06-05 07:16:00,2024-06-05 07:16:00,<NA>,0,0,None,None,%AdjustSeq%\STIMOFunctionTest,Initialized by sequence,...,False,None,None,None,<NA>,False,False,False,False,NaN
4,1315673187101,2024-06-05 07:16:01,2024-06-05 07:16:01,<NA>,0,0,None,None,%AdjustSeq%\STIMOFunctionTest,Initialized by sequence,...,False,None,None,None,<NA>,False,False,False,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1632,1470079434884,2025-01-20 12:48:35,2025-01-20 12:48:37,<NA>,2,0,None,BRAIN,%AdjustSeq%/AdjDicoSeq,AdjDico,...,False,None,None,None,0,False,False,False,False,0.010796
1633,1470079413792,2025-01-20 07:03:49,2025-01-20 07:03:55,<NA>,6,0,None,HEART,%AdjustSeq%/AdjCoilSensSeq,AdjCoilSens,...,False,None,None,None,0,False,False,False,False,0.036113
1634,1470079420677,2025-01-20 09:04:36,2025-01-20 09:04:37,<NA>,1,0,None,BRAIN,%AdjustSeq%/AdjTra1DSeq,AdjTra1D,...,False,None,None,None,0,False,False,False,False,0.006955
1635,1470079438731,2025-01-20 13:40:33,2025-01-20 13:40:34,<NA>,1,0,None,BRAIN,%AdjustSeq%/AdjTra2DSeq,AdjTra2D,...,False,None,None,None,0,False,False,False,False,0.005357


In [11]:
def merge_measurements_parameters(con: dd.DuckDBPyConnection) -> pd.DataFrame:
    """
    Merge the measurements DataFrame with the renamed parameters DataFrame
    using an inner join on MeasurementID and FK_MeasurementID to ensure that
    only measurements with corresponding protocol parameters are included in the merged DataFrame.

    Parameters:
        con (dd.DuckDBPyConnection): A connection object to the DuckDB database.
    Returns:
        pd.DataFrame: A merged DataFrame containing both measurements and protocol parameters.
    """

    # Perform the merging
    measurements_parameters_df = con.execute(
        f"""
        SELECT
            measurements.*, 
            renamed_params.*
        FROM
            measurements
        INNER JOIN
            renamed_params
        ON
            measurements.MeasurementID_meas = renamed_params.FK_MeasurementID_params
        ORDER BY
            measurements.MeasurementID_meas
        """
    ).df()

    # Register the merged DataFrame as a DuckDB table
    con.register("measurements_parameters", measurements_parameters_df)

    return measurements_parameters_df


measurements_parameters_df = merge_measurements_parameters(con)
display(measurements_parameters_df)

,MeasurementID_meas,MeasurementStart_meas,MeasurementEnd_meas,SeqNumMeasPerExam_meas,ScanningTime_meas,NoScanningTime_meas,MeasurementType_meas,BodyRegion_meas,Sequence_meas,Protocol_meas,...,FatSat_params,view sharing_params,Image filter_params,slew rate fast*_params,PositioningMode_params,Dixon Fast_params,2nd TI time_params,Readout segments_params,RPF_params,Local Shim_params
0,1315673197326,2024-06-05 08:24:13,2024-06-05 08:26:56,1,163,0,<NA>,WHOLEBODY,%SiemensSeq%\tse,t2_tirm_cor_GK,...,<NA>,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
1,1315673197754,2024-06-05 08:26:59,2024-06-05 08:26:59,<NA>,0,0,<NA>,WHOLEBODY,%AdjustSeq%/AdjDicoSeq,AdjDico,...,<NA>,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
2,1315673197780,2024-06-05 08:27:00,2024-06-05 08:27:01,<NA>,1,0,<NA>,WHOLEBODY,%AdjustSeq%/AdjFreSeq,AdjFre,...,<NA>,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
3,1315673197803,2024-06-05 08:27:01,2024-06-05 08:27:03,<NA>,2,0,<NA>,WHOLEBODY,%AdjustSeq%/AdjTraSeq,AdjTra,...,<NA>,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
4,1315673197821,2024-06-05 08:27:03,2024-06-05 08:27:04,<NA>,1,0,<NA>,WHOLEBODY,%AdjustSeq%/AdjDicoSeq,AdjDico,...,<NA>,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1344,1470079439259,2025-01-20 13:44:20,2025-01-20 13:46:33,2,133,0,<NA>,BRAIN,%SiemensSeq%\tse,t1_TI900_tra_4mm,...,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4
1345,1470079439374,2025-01-20 13:46:33,2025-01-20 13:47:58,3,85,0,<NA>,BRAIN,%SiemensSeq%\gre,t2_stern_tra_4mm,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1
1346,1470079439457,2025-01-20 13:47:59,2025-01-20 13:50:06,4,127,2,<NA>,BRAIN,%SiemensSeq%\tse,t2_tse_cor_p2_384_3mm,...,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
1347,1470079439561,2025-01-20 13:50:12,2025-01-20 13:52:18,5,126,0,<NA>,BRAIN,%SiemensSeq%\tse,t1_ti900_tra_4mm_KM,...,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4


## Scanner

In [12]:
def get_scanner(data_dir: str) -> pd.DataFrame:
    """
    Get the Scanner.csv files from all MRI folders, join them into a single DataFrame,
    and add a suffix "_scan" to the column names.

    Parameters:
        data_dir (str): Path to the data directory.
    Returns:
        pd.DataFrame: A DataFrame containing the scanner data with a suffix "_scan" added to the column names.
    """

    # Read in the Scanner.csv files where the corresponding
    scanner_df = con.execute(
        f"""
        SELECT
            *
        FROM
            read_csv_auto('{data_dir}/*/*/Scanner.csv', union_by_name=true) AS scanner
        """
    ).df()

    scanner_df = scanner_df.add_suffix("_scan")
    con.register("scanner", scanner_df)

    return scanner_df


scanner_df = get_scanner(data_dir)
display(scanner_df)

,ScannerID_scan,SystemType_scan,SiteName_scan,SiteSecondaryName_scan,SiteStreet_scan,SiteCity_scan,CountryShortName_scan,CountryLongName_scan,Serial_scan,CustomName_scan,Customer_scan,InstallDate_scan,PossNumberExamsPerDay_scan,NumberWorkingDays_scan,NumberWorkingHours_scan
0,52,AvantoFIT,UKT,MR 4,None,Tübingen,DE,Germany,69667,MR51_AvantoFit,Universitätsklinikum Tübingen,2016-07-07,None,False,False
1,52,AvantoFIT,UKT,MR 4,None,Tübingen,DE,Germany,69667,MR51_AvantoFit,Universitätsklinikum Tübingen,2016-07-07,None,False,False
2,52,AvantoFIT,UKT,MR 4,None,Tübingen,DE,Germany,69667,MR51_AvantoFit,Universitätsklinikum Tübingen,2016-07-07,None,False,False
3,52,AvantoFIT,UKT,MR 4,None,Tübingen,DE,Germany,69667,MR51_AvantoFit,Universitätsklinikum Tübingen,2016-07-07,None,False,False
4,55,Vida,UKT,MR 4,None,Tübingen,DE,Germany,75609,MR61_Vida,Universitätsklinikum Tübingen,2016-08-30,None,False,False
5,55,Vida,UKT,MR 4,None,Tübingen,DE,Germany,75609,MR61_Vida,Universitätsklinikum Tübingen,2016-08-30,None,False,False
6,55,Vida,UKT,MR 4,None,Tübingen,DE,Germany,75609,MR61_Vida,Universitätsklinikum Tübingen,2016-08-30,None,False,False
7,55,Vida,UKT,MR 4,None,Tübingen,DE,Germany,75609,MR61_Vida,Universitätsklinikum Tübingen,2016-08-30,None,False,False


In [13]:
def get_examinations(data_dir: str) -> pd.DataFrame:
    """
    Get the Examinations.csv files from all MRI folders, join them into a single DataFrame,
    and add a suffix "_exam" to the column names.

    Parameters:
        data_dir (str): Path to the data directory.
    Returns:
        pd.DataFrame: A DataFrame containing the scanner data with a suffix "_exam" added to the column names.
    """

    # Read in the Scanner.csv files where the corresponding
    examinations_df = con.execute(
        f"""
        SELECT
            *
        FROM
            read_csv_auto('{data_dir}/*/*/Examinations.csv', union_by_name=true) AS examinations
        ORDER BY
            examinations.ExaminationID asc
        """
    ).df()

    # Get the columns that start with "TotalEnergy" which indicate the energy
    # examinations. For each MRI scanner the energy column is differently named
    # but they all start with "TotalEnergy" which allows us to identify them
    energy_columns = [
        col for col in examinations_df.columns if col.startswith("TotalEnergy")
    ]

    # Get all the rows where the TotalEnergy columns are all NaN to check if there
    # are examinations without energy data
    examinations_without_energy = examinations_df[energy_columns].isna().all(axis=1)

    # Add all columns starting with "TotalEnergy" to create a new column "IntermediateEnergy".
    # By default, pandas sum ignores NaN values and sums the available values.
    # However, if all values in a row are NaN, the result will be 0.
    # To ensure the result is NaN when all values are NaN,
    # we identify such rows beforehandand set the sum to NaN for them afterwards.
    examinations_df["SiemensTotalEnergy_KWh"] = examinations_df[energy_columns].sum(
        axis=1, skipna=True
    )

    # Drop all columns starting with "TotalEnergy"
    examinations_df = examinations_df.drop(columns=energy_columns, inplace=False)

    # Set the examinations without energy data to NaN in the SiemensTotalEnergy_KWh column
    examinations_df.loc[examinations_without_energy, "SiemensTotalEnergy_KWh"] = pd.NA

    examinations_df = examinations_df.add_suffix("_exam")
    con.register("examinations", examinations_df)

    return examinations_df


examinations_df = get_examinations(data_dir)
display(examinations_df)

,ExaminationID_exam,Examination_exam,ExaminationStart_exam,ExaminationEnd_exam,Baseline_exam,SWVersion_exam,BodyRegion_exam,Program_exam,LeanExamination_exam,LeanExaminationWoReps_exam,...,FinalQATime_exam,TurnaroundTime_exam,TimeExamStartToDoorClose_exam,TimeDoorOpenToExamEnd_exam,TotalDoorClosedSpan_exam,TimeDoorOpened_exam,TimeDoorCloseToMeasStart_exam,TimeMeasEndToDoorOpen_exam,ScanDuration_ms_exam,SiemensTotalEnergy_KWh_exam
0,3392091,69667_2024-06-05_08:15:46,2024-06-05 08:15:46,2024-06-05 10:25:13,None,None,WHOLEBODY,_unknown,"bolus[2], diff_ep_tse_2d_tra[6], t1_fl_angio_3...","bolus, diff_ep_tse_2d_tra, t1_fl_angio_3d_cor,...",...,88,3447,150,50,7567,981,72,88,7767000,NaN
1,3392092,69667_2024-06-05_11:17:00,2024-06-05 11:17:00,2024-06-05 12:21:21,None,None,LIVER,_unknown,"bolus[2], diff_ep_tse_2d, t1_fl_angio_3d_cor[5...","bolus, diff_ep_tse_2d, t1_fl_angio_3d_cor, t1_...",...,66,3532,182,24,3655,204,20,66,3861000,NaN
2,3392093,69667_2024-06-05_13:09:53,2024-06-05 13:09:53,2024-06-05 14:13:10,None,None,ABDOMENPELVIS,_unknown,"diff_ep_tse_2d, t1_fl_3d_cor, t1_fl_3d_cor_pos...","diff_ep_tse_2d, t1_fl_3d_cor, t1_fl_3d_cor_pos...",...,12,2797,193,31,3573,434,45,12,3797000,NaN
3,3392094,69667_2024-06-05_14:46:42,2024-06-05 14:46:42,2024-06-05 16:22:11,None,None,WHOLEBODY,_unknown,"diff_ep_tse_2d_tra[5], t1_se_tra[2], t1_vibe_d...","diff_ep_tse_2d_tra, t1_se_tra, t1_vibe_dixon_t...",...,73,3431,159,80,5490,519,16,73,5729000,NaN
4,3392095,69667_2024-06-05_17:15:34,2024-06-05 17:15:34,2024-06-05 17:49:24,None,None,LIVER,_unknown,"bolus[2], diff_ep_tse_2d_tra, t1_vibe_dixon_co...","bolus, diff_ep_tse_2d_tra, t1_vibe_dixon_cor, ...",...,81,819,51,23,1956,<NA>,24,81,2030000,NaN
5,3392096,69667_2024-06-05_17:58:24,2024-06-05 17:58:24,2024-06-05 18:38:00,None,None,LIVER,_unknown,"bolus[2], diff_ep_tse_2d_tra, t1_vibe_dixon_co...","bolus, diff_ep_tse_2d_tra, t1_vibe_dixon_cor, ...",...,59,1303,149,-108,2335,80,26,<NA>,2376000,NaN
6,3392097,69667_2024-06-05_18:57:08,2024-06-05 18:57:08,2024-06-05 19:28:46,None,None,LIVER,_unknown,"bolus[2], diff_ep_tse_2d_tra, t1_vibe_dixon_co...","bolus, diff_ep_tse_2d_tra, t1_vibe_dixon_cor, ...",...,67,<NA>,68,63,1897,<NA>,28,67,1898000,NaN
7,4080151,75609_2024-08-29_08:45:11,2024-08-29 08:45:46,2024-08-29 09:27:00,None,None,LIVER,_unknown,"bolus[2], standard, t1_vibe_dixon_cor, t1_vibe...","bolus, standard, t1_vibe_dixon_cor, t1_vibe_di...",...,-54,1761,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2474000,NaN
8,4080152,75609_2024-08-29_09:54:00,2024-08-29 09:54:00,2024-08-29 10:39:46,None,None,NECK,_unknown,"t1_tse_dixon_cor, t1_tse_dixon_tra, t1_tse_tra...","t1_tse_dixon_cor, t1_tse_dixon_tra, t1_tse_tra...",...,0,500,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2746000,NaN
9,4080153,75609_2024-08-29_10:46:40,2024-08-29 10:46:40,2024-08-29 11:07:28,None,None,KNEE,_unknown,"pd_tse_fs_cor, pd_tse_fs_sag, pd_tse_fs_tra, t...","pd_tse_fs_cor, pd_tse_fs_sag, pd_tse_fs_tra, t...",...,0,397,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1248000,NaN


In [14]:
def merge_scanner_measurements_parameters(con: dd.DuckDBPyConnection) -> pd.DataFrame:
    """
    Merge the scanner, examinations, measurements + parameters DataFrames into
    a single DataFrame using inner joins on the relevant IDs to ensure that only
    rows with corresponding data across all three DataFrames are included in
    the merged result. The merging is done based on the relationships between the
    tables, such as ScannerID, ExaminationID, and MeasurementID which is described
    in the Event.csv files.

    Parameters:
        con (dd.DuckDBPyConnection): A connection object to the DuckDB database.
    Returns:
        pd.DataFrame: A merged DataFrame containing scanner, measurements, and protocol parameters data.
    """

    scanner_measurements_parameters_df = con.execute(
        f"""
    SELECT DISTINCT
        events.EventID,
        scanner.*,
        examinations.*,
        measurements_parameters.*
    FROM
        read_csv_auto('{data_dir}/*/*/Events.csv', union_by_name=true) AS events
    INNER JOIN
        scanner
    ON
        events.FK_ScannerID = scanner.ScannerID_scan
    INNER JOIN
        measurements_parameters
    ON
        events.EventID = measurements_parameters.FK_EventID_meas
    INNER JOIN
        examinations
    ON
        events.FK_ExaminationID = examinations.ExaminationID_exam
    WHERE
        events.EventID IS NOT NULL AND
        events.FK_ExaminationID IS NOT NULL AND
        events.FK_ScannerID IS NOT NULL
    ORDER BY
        scanner.SystemType_scan asc,
        examinations.ExaminationID_exam asc,
        measurements_parameters.MeasurementID_meas asc
        
    """
    ).df()

    # Drop the EventID column if not needed
    scanner_measurements_parameters_df.drop(columns=["EventID"], inplace=True)

    # Register the result as a table if needed
    con.register("scanner_measurements_parameters", scanner_measurements_parameters_df)

    return scanner_measurements_parameters_df


scanner_measurements_parameters_df = merge_scanner_measurements_parameters(con)
display(scanner_measurements_parameters_df)

,ScannerID_scan,SystemType_scan,SiteName_scan,SiteSecondaryName_scan,SiteStreet_scan,SiteCity_scan,CountryShortName_scan,CountryLongName_scan,Serial_scan,CustomName_scan,...,FatSat_params,view sharing_params,Image filter_params,slew rate fast*_params,PositioningMode_params,Dixon Fast_params,2nd TI time_params,Readout segments_params,RPF_params,Local Shim_params
0,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,MR51_AvantoFit,...,<NA>,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
1,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,MR51_AvantoFit,...,<NA>,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
2,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,MR51_AvantoFit,...,<NA>,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
3,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,MR51_AvantoFit,...,<NA>,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
4,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,MR51_AvantoFit,...,<NA>,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1273,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,MR61_Vida,...,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4
1274,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,MR61_Vida,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1
1275,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,MR61_Vida,...,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
1276,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,MR61_Vida,...,<NA>,<NA>,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4


# Powerdata

In [15]:
powermeter_scanner_mapping_df

,ScannerID,Serial,PowermeterID
0,52,69667,LQN230413610168
1,55,75609,LQN230413610160
2,55,75609,LQN230413610162


In [16]:
def get_powerdata(data_dir: str) -> pd.DataFrame:
    """
    Get the powerdata_*.csv files from all MRI folders, join them into a single DataFrame,
    and add a suffix "_power" to the column names.

    Parameters:
        data_dir (str): Path to the data directory.
    Returns:
        pd.DataFrame: A DataFrame containing the power data with a suffix "_power" added to the column names.
    """

    power_df = con.execute(
        f"""
        SELECT
            powermeter_scanner_mapping.ScannerID,
            powermeter_scanner_mapping.Serial,
            power.*
        FROM
            read_csv_auto('{data_dir}/*/*/powerdata_*.csv', union_by_name=true) AS power
        INNER JOIN
            powermeter_scanner_mapping
        ON
            power.FK_PowermeterSerial = powermeter_scanner_mapping.PowermeterID
            
        
        """
    ).df()

    con.register("power", power_df)

    return power_df


power_df = get_powerdata(data_dir)
display(power_df)

,ScannerID,Serial,FK_PowermeterSerial,TimeinUTC,VoltageL1L2_V,VoltageL2L3_V,VoltageL3L1_V,CurrentL1_A,CurrentL2_A,CurrentL3_A,...,THDVoltageL2_%,THDVoltageL3_%,DigitalInput,Time,SamplingRate_sec,TotalEnergyL1_KWh,TotalEnergyL2_KWh,TotalEnergyL3_KWh,LeanExamination,TotalEnergy_KWh
0,52,69667,LQN230413610168,2024-11-02 23:00:00,405.95,408.32,407.57,14.29,14.27,14.18,...,3.11,3.26,None,2024-11-03 00:00:00,1.0,0.000732,0.000724,0.000726,idle,0.002181
1,52,69667,LQN230413610168,2024-11-02 23:00:01,405.81,408.22,407.42,14.31,14.29,14.21,...,3.77,3.91,None,2024-11-03 00:00:01,1.0,0.000733,0.000726,0.000728,idle,0.002187
2,52,69667,LQN230413610168,2024-11-02 23:00:02,405.51,408.00,407.18,14.28,14.25,14.18,...,2.92,3.08,None,2024-11-03 00:00:02,1.0,0.000731,0.000723,0.000726,idle,0.002180
3,52,69667,LQN230413610168,2024-11-02 23:00:03,405.42,407.90,407.08,14.24,14.21,14.15,...,2.96,3.11,None,2024-11-03 00:00:03,1.0,0.000727,0.000720,0.000723,idle,0.002171
4,52,69667,LQN230413610168,2024-11-02 23:00:04,405.64,408.06,407.22,14.18,14.15,14.08,...,3.04,3.16,None,2024-11-03 00:00:04,1.0,0.000723,0.000715,0.000718,idle,0.002157
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507505,55,75609,LQN230413610162,2025-01-20 16:34:47,407.00,408.88,407.09,15.49,15.01,14.93,...,2.45,2.42,None,2025-01-20 17:34:47,1.0,0.000906,0.000872,0.000884,idle,0.002662
507506,55,75609,LQN230413610162,2025-01-20 16:34:48,406.98,408.86,407.06,15.52,15.04,14.96,...,2.47,2.46,None,2025-01-20 17:34:48,1.0,0.000909,0.000874,0.000887,idle,0.002670
507507,55,75609,LQN230413610162,2025-01-20 16:34:50,407.05,408.97,407.12,15.52,15.06,14.98,...,2.49,2.52,None,2025-01-20 17:34:50,2.0,0.001817,0.001752,0.001774,idle,0.005343
507508,55,75609,LQN230413610162,2025-01-20 16:34:51,407.02,408.91,407.09,15.57,15.07,15.00,...,2.48,2.45,None,2025-01-20 17:34:51,1.0,0.000912,0.000877,0.000890,idle,0.002678


In [17]:
def get_powerboundries_scanner(data_dir: str) -> pd.DataFrame:
    """
    Get the powerboundries_scanner_mapping.csv file which contains the boundries
    for the different scanner modes which are EcoPowerMode, Idle, Scanning mode.
    The file maps the boundries to the corresponding Serial of the MRI scanner.

    Parameters:
        data_dir (str): Path to the data directory.

    Returns:
        pd.DataFrame: A DataFrame containing the powerboundries for the different scanner modes mapped
                        to the corresponding Serial of the MRI scanner.

    """
    powerboundries_scanner_df = con.execute(
        f"""
        SELECT
            *
        FROM
            read_csv_auto('{data_dir}/powerboundries_scanner_mapping.csv') as powerboundries_scanner
        """
    ).df()

    # Register the scannermode_powermeter DataFrame as a DuckDB table
    con.register("powerboundries_scanner", powerboundries_scanner_df)

    return powerboundries_scanner_df


powerboundries_scanner_df = get_powerboundries_scanner(data_dir)
display(powerboundries_scanner_df)

,Serial,EcoPowerModeBoundary_kW,IdleBoundary_kW
0,69667,11.0,13.0
1,142185,7.0,11.0
2,183811,9.0,10.5
3,202017,12.0,14.0


In [18]:
def enrich_powerdata(power_df: pd.DataFrame) -> pd.DataFrame:
    """
    Enrich the power data by creating new columns for total energy, apparent power,
    active power, and reactive power based on the existing columns. This involves
    summing up the energy and power values across different phases (L1, L2, L3)
    to create unified columns that represent the total values. A date key is
    created from the Time column to allow for daily aggregation of energy data.
    Additionally, calculate the daily energy consumption in the different
    scanner modes (EcoPowerMode, Idle, Scanning) based on the powerboundries
    for the different scanner modes which are mapped to the corresponding
    Serial of the MRI scanner.

    Parameters:
        power_df (pd.DataFrame): The original power data DataFrame to be enriched.

    Returns:
        pd.DataFrame: The enriched power data DataFrame with new columns for total energy,
                    apparent power, active power, reactive power, and a date key
                    for daily aggregation.
    """
    # Create TotalEnergy_KWh by adding up the different phases of
    # TotalEnergyL1_KWh, TotalEnergyL2_KWh, TotalEnergyL3_KWh
    power_df["TotalEnergy_KWh"] = (
        power_df["TotalEnergyL1_KWh"].fillna(0)
        + power_df["TotalEnergyL2_KWh"].fillna(0)
        + power_df["TotalEnergyL3_KWh"].fillna(0)
    )

    # Create TotalApparentPower_KVA by adding up the different phases of
    # ApparentPowerL1_VA, ApparentPowerL2_VA, ApparentPowerL3_VA and
    # converting from VA to KVA
    power_df["TotalApparentPower_KVA"] = (
        power_df["ApparentPowerL1_VA"].fillna(0)
        + power_df["ApparentPowerL2_VA"].fillna(0)
        + power_df["ApparentPowerL3_VA"].fillna(0)
    ) / 1000

    # Create TotalActivePower_KW by adding up the different phases of
    # ActivePowerL1_W, ActivePowerL2_W, ActivePowerL3_W and converting from W to KW
    power_df["TotalActivePower_KW"] = (
        power_df["ActivePowerL1_W"].fillna(0)
        + power_df["ActivePowerL2_W"].fillna(0)
        + power_df["ActivePowerL3_W"].fillna(0)
        # Convert from W to KW
    ) / 1000

    # Create TotalReactivePower_KVAR by taking the square root of the difference between the
    # square of TotalApparentPower_KVA and the square of TotalActivePower_KW
    power_df["TotalReactivePower_KVAR"] = (
        power_df["TotalApparentPower_KVA"] ** 2 - power_df["TotalActivePower_KW"] ** 2
    ).pow(0.5)

    # Create a date key from the Time column to allow for daily aggregation of
    # energy data
    power_df["Date"] = power_df["Time"].dt.date

    # Compute daily total energy by grouping by the powermeter serial and the date
    # and summing the TotalEnergy_KWh
    power_df["DailyTotalEnergy_KWh"] = power_df.groupby(
        ["FK_PowermeterSerial", "Date"]
    )["TotalEnergy_KWh"].transform("sum")

    # Register the powerdata DataFrame as a DuckDB table
    con.register("power", power_df)
    # Perform the join to add EcoPowerModeBoundary_kW and IdleBoundary_kW from powerboundries_scanner
    power_boundries_df = con.execute(
        f"""
        SELECT
            power.*,
            powerboundries_scanner.EcoPowerModeBoundary_kW AS EcoPowerModeBoundary_kW,
            powerboundries_scanner.IdleBoundary_kW AS IdleBoundary_kW
        FROM
            power AS power
        --- Remove the LEFT JOIN and use an INNER JOIN to ensure that only power 
        --- data with corresponding scanner mode information is included.
        --- Right now the powerboundries_scanner is missing some serials which 
        --- would result in dropping a lot of data in the power_df
        LEFT JOIN
            powerboundries_scanner
        ON
            power.Serial = powerboundries_scanner.Serial
        ORDER BY
            power.ScannerID asc,
            power.Serial asc,
            power.Time asc
        """
    ).df()

    # Register the final enriched powerdata DataFrame as a DuckDB table
    con.register("power_boundries", power_boundries_df)

    return power_boundries_df


power_boundries_df = enrich_powerdata(power_df)
display(power_boundries_df)

,ScannerID,Serial,FK_PowermeterSerial,TimeinUTC,VoltageL1L2_V,VoltageL2L3_V,VoltageL3L1_V,CurrentL1_A,CurrentL2_A,CurrentL3_A,...,TotalEnergyL3_KWh,LeanExamination,TotalEnergy_KWh,TotalApparentPower_KVA,TotalActivePower_KW,TotalReactivePower_KVAR,Date,DailyTotalEnergy_KWh,EcoPowerModeBoundary_kW,IdleBoundary_kW
0,52,69667,LQN230413610168,2024-11-02 23:00:00,405.95,408.32,407.57,14.29,14.27,14.18,...,0.000726,idle,0.002181,10.05114,7.85265,6.273859,2024-11-03,150.966201,11.0,13.0
1,52,69667,LQN230413610168,2024-11-02 23:00:01,405.81,408.22,407.42,14.31,14.29,14.21,...,0.000728,idle,0.002187,10.06624,7.87314,6.272388,2024-11-03,150.966201,11.0,13.0
2,52,69667,LQN230413610168,2024-11-02 23:00:02,405.51,408.00,407.18,14.28,14.25,14.18,...,0.000726,idle,0.002180,10.03273,7.84716,6.251220,2024-11-03,150.966201,11.0,13.0
3,52,69667,LQN230413610168,2024-11-02 23:00:03,405.42,407.90,407.08,14.24,14.21,14.15,...,0.000723,idle,0.002171,10.00489,7.81436,6.247688,2024-11-03,150.966201,11.0,13.0
4,52,69667,LQN230413610168,2024-11-02 23:00:04,405.64,408.06,407.22,14.18,14.15,14.08,...,0.000718,idle,0.002157,9.96398,7.76398,6.245119,2024-11-03,150.966201,11.0,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507505,55,75609,LQN230413610160,2025-01-20 22:59:57,404.72,407.12,406.64,3.78,3.89,5.43,...,0.000083,idle,0.000196,3.07648,0.70679,2.994191,2025-01-20,119.254477,NaN,NaN
507506,55,75609,LQN230413610162,2025-01-20 22:59:58,405.46,407.60,406.36,15.60,14.99,15.00,...,0.000892,idle,0.002674,10.69729,9.62759,4.662781,2025-01-20,236.503533,NaN,NaN
507507,55,75609,LQN230413610160,2025-01-20 22:59:58,404.56,406.94,406.46,3.77,3.88,5.40,...,0.000081,idle,0.000193,3.05996,0.69421,2.980172,2025-01-20,119.254477,NaN,NaN
507508,55,75609,LQN230413610162,2025-01-20 22:59:59,405.44,407.58,406.32,15.59,14.98,14.99,...,0.000891,idle,0.002671,10.69084,9.61705,4.669733,2025-01-20,236.503533,NaN,NaN


In [19]:
def calculate_daily_energy() -> pd.DataFrame:
    """
    Calculate the daily idle energy and daily scan energy per powermeter by
    using DuckDB to perform the necessary aggregations and joins.
    The daily idle energy is calculated by summing the TotalEnergy_KWh for
    rows where LeanExamination is 'idle' grouped by FK_PowermeterSerial
    and Date. The daily scan energy is then calculated by subtracting the
    daily idle energy from the daily total energy.

    Returns:
        pd.DataFrame: A DataFrame containing the original power data along
                        with the calculated daily idle energy and daily scan
                        energy per powermeter.
    """

    # Compute aggregated eco energy (one row per group)
    eco_agg = con.execute(
        f"""
        SELECT
            power_boundries.FK_PowermeterSerial,
            power_boundries.Date,
            power_boundries.Serial,
            SUM(power_boundries.TotalEnergy_KWh) AS DailyEcoPowerModeEnergy_KWh
        FROM
            power_boundries
        WHERE 
            power_boundries.LeanExamination = 'idle' AND
            power_boundries.TotalActivePower_KW < power_boundries.EcoPowerModeBoundary_kW
        GROUP BY
            power_boundries.Serial,
            power_boundries.FK_PowermeterSerial, 
            power_boundries.Date
        """
    ).df()

    # Compute aggregated idle energy (one row per group)
    idle_agg = con.execute(
        f"""
        SELECT
            power_boundries.FK_PowermeterSerial,
            power_boundries.Date,
            power_boundries.Serial,
            SUM(power_boundries.TotalEnergy_KWh) AS DailyIdleEnergy_KWh
        FROM
            power_boundries
        WHERE 
            power_boundries.LeanExamination = 'idle' AND
            power_boundries.TotalActivePower_KW BETWEEN power_boundries.EcoPowerModeBoundary_kW AND power_boundries.IdleBoundary_kW
        GROUP BY
            power_boundries.Serial,
            power_boundries.FK_PowermeterSerial, 
            power_boundries.Date
        """
    ).df()

    # Merge the aggregated eco energy and idle energy data on Serial, FK_PowermeterSerial,
    # and Date using an outer join to ensure that all combinations are included,
    # even if one of the energy types is missing. Fill NaN values with 0 after
    # the merge to indicate that if there is no energy recorded for a type, it
    # should be treated as 0.
    combined_agg = pd.merge(
        idle_agg, eco_agg, on=["Serial", "FK_PowermeterSerial", "Date"], how="outer"
    ).fillna(0)

    # Register the combined aggregated data
    con.register("combined_agg", combined_agg)

    # LEFT JOIN the combined aggregated data back to power_boundries
    power_daily_df = con.execute(
        f"""
        SELECT
            power_boundries.*,
            combined_agg.DailyEcoPowerModeEnergy_KWh,
            combined_agg.DailyIdleEnergy_KWh
        FROM
            power_boundries
        LEFT JOIN 
            combined_agg
        ON 
            combined_agg.FK_PowermeterSerial = power_boundries.FK_PowermeterSerial AND
            combined_agg.Date = power_boundries.Date AND
            combined_agg.Serial = power_boundries.Serial
        ORDER BY
            power_boundries.ScannerID asc,
            power_boundries.Time asc,
            power_boundries.FK_PowermeterSerial asc
        """
    ).df()

    # Calculate the DailyScanEnergy_KWh by subtracting the
    # DailyIdleEnergy_KWh and DailyEcoPowerModeEnergy_KWh from the DailyTotalEnergy_KWh
    power_daily_df["DailyScanEnergy_KWh"] = (
        power_daily_df["DailyTotalEnergy_KWh"]
        - power_daily_df["DailyIdleEnergy_KWh"].fillna(0)
        - power_daily_df["DailyEcoPowerModeEnergy_KWh"].fillna(0)
    ).fillna(0)

    # Add a suffix to the column names "_energy"
    power_daily_df = power_daily_df.add_suffix("_energy")

    # Register the result as a table if needed
    con.register("power", power_daily_df)

    return power_daily_df


power_daily_df = calculate_daily_energy()
display(power_daily_df)

,ScannerID_energy,Serial_energy,FK_PowermeterSerial_energy,TimeinUTC_energy,VoltageL1L2_V_energy,VoltageL2L3_V_energy,VoltageL3L1_V_energy,CurrentL1_A_energy,CurrentL2_A_energy,CurrentL3_A_energy,...,TotalApparentPower_KVA_energy,TotalActivePower_KW_energy,TotalReactivePower_KVAR_energy,Date_energy,DailyTotalEnergy_KWh_energy,EcoPowerModeBoundary_kW_energy,IdleBoundary_kW_energy,DailyEcoPowerModeEnergy_KWh_energy,DailyIdleEnergy_KWh_energy,DailyScanEnergy_KWh_energy
0,52,69667,LQN230413610168,2024-11-02 23:00:00,405.95,408.32,407.57,14.29,14.27,14.18,...,10.05114,7.85265,6.273859,2024-11-03,150.966201,11.0,13.0,150.966201,0.0,0.000000
1,52,69667,LQN230413610168,2024-11-02 23:00:01,405.81,408.22,407.42,14.31,14.29,14.21,...,10.06624,7.87314,6.272388,2024-11-03,150.966201,11.0,13.0,150.966201,0.0,0.000000
2,52,69667,LQN230413610168,2024-11-02 23:00:02,405.51,408.00,407.18,14.28,14.25,14.18,...,10.03273,7.84716,6.251220,2024-11-03,150.966201,11.0,13.0,150.966201,0.0,0.000000
3,52,69667,LQN230413610168,2024-11-02 23:00:03,405.42,407.90,407.08,14.24,14.21,14.15,...,10.00489,7.81436,6.247688,2024-11-03,150.966201,11.0,13.0,150.966201,0.0,0.000000
4,52,69667,LQN230413610168,2024-11-02 23:00:04,405.64,408.06,407.22,14.18,14.15,14.08,...,9.96398,7.76398,6.245119,2024-11-03,150.966201,11.0,13.0,150.966201,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507505,55,75609,LQN230413610162,2025-01-20 22:59:57,405.66,407.85,406.60,15.73,15.14,15.16,...,10.80788,9.73542,4.693812,2025-01-20,236.503533,NaN,NaN,NaN,NaN,236.503533
507506,55,75609,LQN230413610160,2025-01-20 22:59:58,404.56,406.94,406.46,3.77,3.88,5.40,...,3.05996,0.69421,2.980172,2025-01-20,119.254477,NaN,NaN,NaN,NaN,119.254477
507507,55,75609,LQN230413610162,2025-01-20 22:59:58,405.46,407.60,406.36,15.60,14.99,15.00,...,10.69729,9.62759,4.662781,2025-01-20,236.503533,NaN,NaN,NaN,NaN,236.503533
507508,55,75609,LQN230413610160,2025-01-20 22:59:59,404.46,406.90,406.40,3.77,3.88,5.41,...,3.06273,0.70054,2.981536,2025-01-20,119.254477,NaN,NaN,NaN,NaN,119.254477


In [20]:
def create_power_daily_df() -> pd.DataFrame:
    """
    Drop the duplicates in the power_df to create a daily aggregated power
    data DataFrame with one row per powermeter and day. This is done
    by dropping duplicates based on the FK_PowermeterSerial and Date columns,
    which represent the unique combination of powermeter and day.
    The resulting DataFrame will contain the daily total energy,
    daily idle energy, daily eco power mode energy, and daily scan
    energy for each powermeter.

    """

    power_daily_df = con.execute(
        f"""
        SELECT DISTINCT
            FK_PowermeterSerial_energy,
            Date_energy,
            Serial_energy,
            DailyTotalEnergy_KWh_energy,
            DailyIdleEnergy_KWh_energy,
            DailyEcoPowerModeEnergy_KWh_energy,
            DailyScanEnergy_KWh_energy
        FROM
            power
        ORDER BY
            Serial_energy asc,
            FK_PowermeterSerial_energy asc,
            Date_energy asc
        """
    ).df()

    # Register the daily aggregated power data DataFrame as a DuckDB table
    con.register("power_daily", power_daily_df)

    return power_daily_df


power_daily_df = create_power_daily_df()
display(power_daily_df)

,FK_PowermeterSerial_energy,Date_energy,Serial_energy,DailyTotalEnergy_KWh_energy,DailyIdleEnergy_KWh_energy,DailyEcoPowerModeEnergy_KWh_energy,DailyScanEnergy_KWh_energy
0,LQN230413610168,2024-11-03,69667,150.966201,0.000000,150.966201,0.000000
1,LQN230413610168,2024-12-02,69667,323.334785,161.868608,24.153230,137.312947
2,LQN230413610160,2024-06-02,75609,-0.270318,NaN,NaN,-0.270318
3,LQN230413610160,2025-01-20,75609,119.254477,NaN,NaN,119.254477
4,LQN230413610162,2024-06-02,75609,145.102424,NaN,NaN,145.102424
5,LQN230413610162,2025-01-20,75609,236.503533,NaN,NaN,236.503533


## Merging Powerdata

In [21]:
def merge_scanner_powermeterID(data_dir: str) -> pd.DataFrame:
    """
    Merge the powermeterID from the mapping CSV file into the scanner measurements
    parameters DataFrame to know which powermeter corresponds to which scanner.
    The numbers of rows can increase due to the fact that there can be multiple
    powermeter data files per scanner which should all be included in the merged DataFrame.


    Parameters:
        data_dir (str): Path to the data directory.

    Returns:
        pd.DataFrame: A DataFrame containing the scanner measurements parameters
                        along with the corresponding PowermeterID.
    """

    ID_scanner_measurements_parameters_df = con.execute(
        f"""
    SELECT
        powermeter_scanner_mapping.PowermeterID AS PowermeterID_scan,
        scanner_measurements_parameters.*
    FROM
        scanner_measurements_parameters
    INNER JOIN
        read_csv_auto('{data_dir}/powermeter_scanner_mapping.csv') AS powermeter_scanner_mapping
    ON
        scanner_measurements_parameters.ScannerID_scan = powermeter_scanner_mapping.ScannerID
    """
    ).df()

    # Create a Date column from the MeasurementStart_meas column to allow
    # for easier merging with the power data which is on a daily level
    ID_scanner_measurements_parameters_df["Date_energy"] = (
        ID_scanner_measurements_parameters_df["MeasurementStart_meas"].dt.date
    )

    # Register the updated DataFrame as a DuckDB table
    con.register(
        "ID_scanner_measurements_parameters",
        ID_scanner_measurements_parameters_df,
    )

    return ID_scanner_measurements_parameters_df


ID_scanner_measurements_parameters_df = merge_scanner_powermeterID(data_dir)
display(ID_scanner_measurements_parameters_df)

,PowermeterID_scan,ScannerID_scan,SystemType_scan,SiteName_scan,SiteSecondaryName_scan,SiteStreet_scan,SiteCity_scan,CountryShortName_scan,CountryLongName_scan,Serial_scan,...,view sharing_params,Image filter_params,slew rate fast*_params,PositioningMode_params,Dixon Fast_params,2nd TI time_params,Readout segments_params,RPF_params,Local Shim_params,Date_energy
0,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>,2024-06-05
1,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>,2024-06-05
2,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>,2024-06-05
3,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>,2024-06-05
4,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,<NA>,1,2,1,<NA>,<NA>,<NA>,<NA>,<NA>,2024-06-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1875,LQN230413610160,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,<NA>,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4,2025-01-20
1876,LQN230413610160,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,2025-01-20
1877,LQN230413610160,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,<NA>,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2,2025-01-20
1878,LQN230413610160,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,<NA>,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,4,2025-01-20


In [22]:
power_df

,ScannerID,Serial,FK_PowermeterSerial,TimeinUTC,VoltageL1L2_V,VoltageL2L3_V,VoltageL3L1_V,CurrentL1_A,CurrentL2_A,CurrentL3_A,...,TotalEnergyL1_KWh,TotalEnergyL2_KWh,TotalEnergyL3_KWh,LeanExamination,TotalEnergy_KWh,TotalApparentPower_KVA,TotalActivePower_KW,TotalReactivePower_KVAR,Date,DailyTotalEnergy_KWh
0,52,69667,LQN230413610168,2024-11-02 23:00:00,405.95,408.32,407.57,14.29,14.27,14.18,...,0.000732,0.000724,0.000726,idle,0.002181,10.05114,7.85265,6.273859,2024-11-03,150.966201
1,52,69667,LQN230413610168,2024-11-02 23:00:01,405.81,408.22,407.42,14.31,14.29,14.21,...,0.000733,0.000726,0.000728,idle,0.002187,10.06624,7.87314,6.272388,2024-11-03,150.966201
2,52,69667,LQN230413610168,2024-11-02 23:00:02,405.51,408.00,407.18,14.28,14.25,14.18,...,0.000731,0.000723,0.000726,idle,0.002180,10.03273,7.84716,6.251220,2024-11-03,150.966201
3,52,69667,LQN230413610168,2024-11-02 23:00:03,405.42,407.90,407.08,14.24,14.21,14.15,...,0.000727,0.000720,0.000723,idle,0.002171,10.00489,7.81436,6.247688,2024-11-03,150.966201
4,52,69667,LQN230413610168,2024-11-02 23:00:04,405.64,408.06,407.22,14.18,14.15,14.08,...,0.000723,0.000715,0.000718,idle,0.002157,9.96398,7.76398,6.245119,2024-11-03,150.966201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507505,55,75609,LQN230413610162,2025-01-20 16:34:47,407.00,408.88,407.09,15.49,15.01,14.93,...,0.000906,0.000872,0.000884,idle,0.002662,10.69049,9.58435,4.735695,2025-01-20,236.503533
507506,55,75609,LQN230413610162,2025-01-20 16:34:48,406.98,408.86,407.06,15.52,15.04,14.96,...,0.000909,0.000874,0.000887,idle,0.002670,10.71251,9.61059,4.732275,2025-01-20,236.503533
507507,55,75609,LQN230413610162,2025-01-20 16:34:50,407.05,408.97,407.12,15.52,15.06,14.98,...,0.001817,0.001752,0.001774,idle,0.005343,10.72452,9.61797,4.744469,2025-01-20,236.503533
507508,55,75609,LQN230413610162,2025-01-20 16:34:51,407.02,408.91,407.09,15.57,15.07,15.00,...,0.000912,0.000877,0.000890,idle,0.002678,10.74146,9.64046,4.737140,2025-01-20,236.503533


In [23]:
def get_measurement_power() -> pd.DataFrame:
    """
    Merge the energy and power data into the scanner measurements parameters
    DataFrame by aggregating the power data for each measurement period defined by
    MeasurementStart_meas and MeasurementEnd_meas. The aggregation includes
    calculating the total energy and mean power values during the measurement
    periods for each scanner. This allows us to associate the relevant power data
    with each scanner measurement. Merge the data back to the original
    scanner measurements parameters DataFrame to keep all rows even if no
    power data is available.

    Only perform the join if TotalEnergy_KWh_energy exists in this interval and if the FK_PowermeterSerial_energy exists in the file.

    Returns:
        pd.DataFrame: A DataFrame containing the scanner measurements parameters
                        along with the aggregated energy and power data for each
                        measurement period.
    """

    measurements_power_df = con.execute(
        f"""
    WITH aggregated_data AS (
        SELECT
            IDsmp.ScannerID_scan,
            IDsmp.MeasurementID_meas,
            IDsmp.PowermeterID_scan,
            SUM(power.TotalEnergy_KWh_energy) AS TotalEnergy_KWh_meas,
            AVG(power.TotalActivePower_KW_energy) AS TotalActivePower_KW_meas,
            AVG(power.TotalApparentPower_KVA_energy) AS TotalApparentPower_KVA_meas,
            AVG(power.TotalReactivePower_KVAR_energy) AS TotalReactivePower_KVAR_meas
        FROM 
            ID_scanner_measurements_parameters AS IDsmp
        INNER JOIN 
            power
        ON 
            IDsmp.PowermeterID_scan = power.FK_PowermeterSerial_energy
            AND power.Time_energy BETWEEN IDsmp.MeasurementStart_meas AND IDsmp.MeasurementEnd_meas
        GROUP BY 
            IDsmp.MeasurementID_meas, 
            IDsmp.ScannerID_scan,
            IDsmp.PowermeterID_scan
    )
    -- Attach the aggregated results back to each measurement keeps rows without power data
    SELECT
        IDsmp.ScannerID_scan,
        IDsmp.PowermeterID_scan,
        IDsmp.MeasurementID_meas,   
        aggregated_data.TotalEnergy_KWh_meas AS TotalEnergy_KWh_meas,
        aggregated_data.TotalActivePower_KW_meas AS TotalActivePower_KW_meas,
        aggregated_data.TotalApparentPower_KVA_meas AS TotalApparentPower_KVA_meas,
        aggregated_data.TotalReactivePower_KVAR_meas AS TotalReactivePower_KVAR_meas
    FROM 
        ID_scanner_measurements_parameters AS IDsmp
    LEFT JOIN 
        aggregated_data AS aggregated_data
    ON 
        aggregated_data.ScannerID_scan = IDsmp.ScannerID_scan AND
        aggregated_data.MeasurementID_meas = IDsmp.MeasurementID_meas AND
        aggregated_data.PowermeterID_scan = IDsmp.PowermeterID_scan
    WHERE
        aggregated_data.TotalEnergy_KWh_meas IS NOT NULL
    ORDER BY
        IDsmp.ScannerID_scan asc,
        IDsmp.MeasurementID_meas asc,
        IDsmp.PowermeterID_scan asc
    """
    ).df()

    # Register the DataFrame as a DuckDB table
    con.register("measurements_power", measurements_power_df)

    return measurements_power_df


measurements_power_df = get_measurement_power()
display(measurements_power_df)

,ScannerID_scan,PowermeterID_scan,MeasurementID_meas,TotalEnergy_KWh_meas,TotalActivePower_KW_meas,TotalApparentPower_KVA_meas,TotalReactivePower_KVAR_meas
0,52,LQN230413610168,1437888325575,0.637343,18.013544,21.365114,11.463432
1,52,LQN230413610168,1437888325695,0.004113,14.806590,17.520260,9.366131
2,52,LQN230413610168,1437888325716,0.004128,14.859760,17.582910,9.399269
3,52,LQN230413610168,1437888325737,0.019292,23.150973,26.514657,12.897089
4,52,LQN230413610168,1437888325759,0.008857,31.883980,35.576720,15.783372
...,...,...,...,...,...,...,...
631,55,LQN230413610162,1470079439457,0.510795,14.365759,16.260224,7.616257
632,55,LQN230413610160,1470079439561,0.478851,13.529552,17.714101,11.413250
633,55,LQN230413610162,1470079439561,0.494072,13.962232,15.917473,7.636701
634,55,LQN230413610160,1470079439643,1.268129,16.928325,21.668664,13.505508


In [24]:
def get_examination_power() -> pd.DataFrame:
    """
    Merge the energy and power data into the scanner examinations parameters
    DataFrame by aggregating the power data for each examination period defined by
    ExaminationStart_exam and ExaminationEnd_exam. The aggregation includes
    calculating the total energy and mean power values during the examination
    periods for each scanner. This allows us to associate the relevant power data
    with each scanner examination. Merge the data back to the original
    scanner examinations parameters DataFrame to keep all rows even if no
    power data is available.

    Only perform the join if TotalEnergy_KWh_energy exists in this interval and if the FK_PowermeterSerial_energy exists in the file.

    Returns:
        pd.DataFrame: A DataFrame containing the scanner examinations parameters
                      along with the aggregated energy and power data for each
                      examination period.
    """

    # Get unique combinations of ScannerID, ExaminationID, and PowermeterID to
    # avoid duplicate rows in the result due to multiple measurements per examination
    unique_ID_scanner_measurements_parameters = con.execute(
        f"""
        SELECT 
            *
        FROM 
            ID_scanner_measurements_parameters
        QUALIFY 
            ROW_NUMBER() 
        OVER 
            (PARTITION BY ScannerID_scan, ExaminationID_exam, PowermeterID_scan) = 1
        """
    ).df()

    # Register the unique DataFrame as a DuckDB table
    con.register(
        "unique_ID_scanner_measurements_parameters",
        unique_ID_scanner_measurements_parameters,
    )

    examinations_power_df = con.execute(
        f"""
    WITH aggregated_data AS (
        SELECT
            IDsmp.ScannerID_scan,
            IDsmp.ExaminationID_exam,
            IDsmp.PowermeterID_scan,
            SUM(power.TotalEnergy_KWh_energy) AS TotalEnergy_KWh_exam,
            AVG(power.TotalActivePower_KW_energy) AS TotalActivePower_KW_exam,
            AVG(power.TotalApparentPower_KVA_energy) AS TotalApparentPower_KVA_exam,
            AVG(power.TotalReactivePower_KVAR_energy) AS TotalReactivePower_KVAR_exam
        FROM 
            unique_ID_scanner_measurements_parameters AS IDsmp
        INNER JOIN 
            power
        ON 
            IDsmp.PowermeterID_scan = power.FK_PowermeterSerial_energy
            AND power.Time_energy BETWEEN IDsmp.ExaminationStart_exam AND IDsmp.ExaminationEnd_exam
        GROUP BY 
            IDsmp.ScannerID_scan,
            IDsmp.ExaminationID_exam, 
            IDsmp.PowermeterID_scan
            
    )
    -- Attach the aggregated results back to each examination, keeps rows without power data
    SELECT
        IDsmp.ScannerID_scan,
        IDsmp.PowermeterID_scan,
        IDsmp.ExaminationID_exam,
        aggregated_data.TotalEnergy_KWh_exam AS TotalEnergy_KWh_exam,
        aggregated_data.TotalActivePower_KW_exam AS TotalActivePower_KW_exam,
        aggregated_data.TotalApparentPower_KVA_exam AS TotalApparentPower_KVA_exam,
        aggregated_data.TotalReactivePower_KVAR_exam AS TotalReactivePower_KVAR_exam
    FROM 
        unique_ID_scanner_measurements_parameters AS IDsmp
    LEFT JOIN 
        aggregated_data AS aggregated_data
    ON 
        aggregated_data.ScannerID_scan = IDsmp.ScannerID_scan AND
        aggregated_data.ExaminationID_exam = IDsmp.ExaminationID_exam AND
        aggregated_data.PowermeterID_scan = IDsmp.PowermeterID_scan
    WHERE
        aggregated_data.TotalEnergy_KWh_exam IS NOT NULL
    ORDER BY
        IDsmp.ScannerID_scan asc,
        IDsmp.ExaminationID_exam asc,
        IDsmp.PowermeterID_scan asc
    """
    ).df()

    # Register the DataFrame as a DuckDB table
    con.register("examinations_power", examinations_power_df)

    return examinations_power_df


examinations_power_df = get_examination_power()
display(examinations_power_df)

,ScannerID_scan,PowermeterID_scan,ExaminationID_exam,TotalEnergy_KWh_exam,TotalActivePower_KW_exam,TotalApparentPower_KVA_exam,TotalReactivePower_KVAR_exam
0,52,LQN230413610168,4764007,13.534349,18.841875,22.364046,12.024971
1,52,LQN230413610168,4764008,8.720136,18.226021,21.540895,11.463270
2,52,LQN230413610168,4764009,15.720189,21.540128,25.574502,13.762480
3,52,LQN230413610168,4764011,12.047036,20.098337,23.887145,12.851608
4,52,LQN230413610168,4764012,11.550374,22.299544,26.512067,14.322110
5,52,LQN230413610168,4764013,10.071896,21.509951,25.622188,13.902926
6,52,LQN230413610168,4764014,7.994805,20.296960,24.454229,13.578452
7,55,LQN230413610160,4952958,8.491260,13.058342,17.616999,11.662556
8,55,LQN230413610162,4952958,7.654888,11.769633,13.119547,5.788132
9,55,LQN230413610160,4952959,9.344503,21.916481,27.502883,16.457252


In [25]:
def merge_measurements_examinations_power() -> pd.DataFrame:
    """
    Merge the aggregated power data for measurements and examinations back to the
    original ID_scanner_measurements_parameters DataFrame to have a complete view of
    all the scanner measurements parameters along with the corresponding power data.
    This merge is done using LEFT JOINs to ensure that all rows from the original
    DataFrame are retained even if there is no corresponding power data.

    Returns:
        pd.DataFrame: A DataFrame containing the scanner measurements parameters
                      along with the corresponding aggregated power data for both
                      measurements and examinations.
    """

    measurements_examinations_power_df = con.execute(
        f"""
        SELECT
            IDsmp.*,
            examinations_power.TotalEnergy_KWh_exam,
            examinations_power.TotalActivePower_KW_exam,
            examinations_power.TotalApparentPower_KVA_exam,
            examinations_power.TotalReactivePower_KVAR_exam,
            measurements_power.TotalEnergy_KWh_meas,
            measurements_power.TotalActivePower_KW_meas,
            measurements_power.TotalApparentPower_KVA_meas,
            measurements_power.TotalReactivePower_KVAR_meas
        FROM
            ID_scanner_measurements_parameters AS IDsmp
        INNER JOIN 
            measurements_power
        ON 
            IDsmp.ScannerID_scan = measurements_power.ScannerID_scan AND
            IDsmp.MeasurementID_meas = measurements_power.MeasurementID_meas AND
            IDsmp.PowermeterID_scan = measurements_power.PowermeterID_scan
        INNER JOIN 
            examinations_power
        ON 
            IDsmp.ScannerID_scan = examinations_power.ScannerID_scan AND
            IDsmp.ExaminationID_exam = examinations_power.ExaminationID_exam AND
            IDsmp.PowermeterID_scan = examinations_power.PowermeterID_scan
        ORDER BY
            IDsmp.ScannerID_scan asc,
            IDsmp.PowermeterID_scan asc,
            IDsmp.MeasurementID_meas asc,
            IDsmp.ExaminationID_exam asc
        """
    ).df()

    # Register the merged DataFrame as a DuckDB table if needed
    con.register("measurements_examinations_power", measurements_examinations_power_df)

    return measurements_examinations_power_df


measurements_examinations_power_df = merge_measurements_examinations_power()
display(measurements_examinations_power_df)

,PowermeterID_scan,ScannerID_scan,SystemType_scan,SiteName_scan,SiteSecondaryName_scan,SiteStreet_scan,SiteCity_scan,CountryShortName_scan,CountryLongName_scan,Serial_scan,...,Local Shim_params,Date_energy,TotalEnergy_KWh_exam,TotalActivePower_KW_exam,TotalApparentPower_KVA_exam,TotalReactivePower_KVAR_exam,TotalEnergy_KWh_meas,TotalActivePower_KW_meas,TotalApparentPower_KVA_meas,TotalReactivePower_KVAR_meas
0,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,<NA>,2024-12-02,12.047036,20.098337,23.887145,12.851608,0.637343,18.013544,21.365114,11.463432
1,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,<NA>,2024-12-02,12.047036,20.098337,23.887145,12.851608,0.004113,14.806590,17.520260,9.366131
2,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,<NA>,2024-12-02,12.047036,20.098337,23.887145,12.851608,0.004128,14.859760,17.582910,9.399269
3,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,<NA>,2024-12-02,12.047036,20.098337,23.887145,12.851608,0.019292,23.150973,26.514657,12.897089
4,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,<NA>,2024-12-02,12.047036,20.098337,23.887145,12.851608,0.008857,31.883980,35.576720,15.783372
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
631,LQN230413610162,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,4,2025-01-20,3.714339,12.783171,14.365418,6.543361,0.514964,13.750943,15.649125,7.465698
632,LQN230413610162,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,1,2025-01-20,3.714339,12.783171,14.365418,6.543361,0.274517,11.491171,12.704864,5.418419
633,LQN230413610162,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,2,2025-01-20,3.714339,12.783171,14.365418,6.543361,0.510795,14.365759,16.260224,7.616257
634,LQN230413610162,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,4,2025-01-20,3.714339,12.783171,14.365418,6.543361,0.494072,13.962232,15.917473,7.636701


In [26]:
power_daily_df

,FK_PowermeterSerial_energy,Date_energy,Serial_energy,DailyTotalEnergy_KWh_energy,DailyIdleEnergy_KWh_energy,DailyEcoPowerModeEnergy_KWh_energy,DailyScanEnergy_KWh_energy
0,LQN230413610168,2024-11-03,69667,150.966201,0.000000,150.966201,0.000000
1,LQN230413610168,2024-12-02,69667,323.334785,161.868608,24.153230,137.312947
2,LQN230413610160,2024-06-02,75609,-0.270318,NaN,NaN,-0.270318
3,LQN230413610160,2025-01-20,75609,119.254477,NaN,NaN,119.254477
4,LQN230413610162,2024-06-02,75609,145.102424,NaN,NaN,145.102424
5,LQN230413610162,2025-01-20,75609,236.503533,NaN,NaN,236.503533


In [27]:
def merge_power_daily() -> pd.DataFrame:
    """
    Map the daily energy consumption in the different scanner modes (EcoPowerMode, Idle, Scanning, Total)
    from the power_daily_df DataFrame to the corresponding measurements using
    the Seriel, FK_PowermeterSerial and the date of the measurement.
    This allows us to have the daily energy consumption in the different
    scanner modes associated with each measurement, which can be useful for
    analysis and understanding the energy usage patterns during the measurements.

    Returns:
        pd.DataFrame: A DataFrame containing the scanner measurements parameters
                      along with the corresponding daily energy consumption in the different
                      scanner modes for each measurement.
    """
    power_daily_scannerInfo_df = con.execute(
        f"""
        SELECT
            mep.*,
            power_daily.DailyEcoPowerModeEnergy_KWh_energy,
            power_daily.DailyIdleEnergy_KWh_energy,
            power_daily.DailyScanEnergy_KWh_energy,
            power_daily.DailyTotalEnergy_KWh_energy
        FROM
            measurements_examinations_power as mep
        LEFT JOIN 
            power_daily
        ON 
            mep.Serial_scan = power_daily.Serial_energy AND
            mep.PowermeterID_scan = power_daily.FK_PowermeterSerial_energy AND
            mep.Date_energy = power_daily.Date_energy
        ORDER BY
            mep.ScannerID_scan asc,
            mep.PowermeterID_scan asc,
            mep.MeasurementID_meas asc,
            mep.ExaminationID_exam asc
        """
    ).df()

    # Register the merged DataFrame as a DuckDB table if needed
    con.register("power_daily_scannerInfo", power_daily_scannerInfo_df)

    return power_daily_scannerInfo_df


power_daily_scannerInfo_df = merge_power_daily()
display(power_daily_scannerInfo_df)

,PowermeterID_scan,ScannerID_scan,SystemType_scan,SiteName_scan,SiteSecondaryName_scan,SiteStreet_scan,SiteCity_scan,CountryShortName_scan,CountryLongName_scan,Serial_scan,...,TotalApparentPower_KVA_exam,TotalReactivePower_KVAR_exam,TotalEnergy_KWh_meas,TotalActivePower_KW_meas,TotalApparentPower_KVA_meas,TotalReactivePower_KVAR_meas,DailyEcoPowerModeEnergy_KWh_energy,DailyIdleEnergy_KWh_energy,DailyScanEnergy_KWh_energy,DailyTotalEnergy_KWh_energy
0,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,23.887145,12.851608,0.637343,18.013544,21.365114,11.463432,24.15323,161.868608,137.312947,323.334785
1,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,23.887145,12.851608,0.004113,14.806590,17.520260,9.366131,24.15323,161.868608,137.312947,323.334785
2,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,23.887145,12.851608,0.004128,14.859760,17.582910,9.399269,24.15323,161.868608,137.312947,323.334785
3,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,23.887145,12.851608,0.019292,23.150973,26.514657,12.897089,24.15323,161.868608,137.312947,323.334785
4,LQN230413610168,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,...,23.887145,12.851608,0.008857,31.883980,35.576720,15.783372,24.15323,161.868608,137.312947,323.334785
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
631,LQN230413610162,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,14.365418,6.543361,0.514964,13.750943,15.649125,7.465698,NaN,NaN,236.503533,236.503533
632,LQN230413610162,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,14.365418,6.543361,0.274517,11.491171,12.704864,5.418419,NaN,NaN,236.503533,236.503533
633,LQN230413610162,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,14.365418,6.543361,0.510795,14.365759,16.260224,7.616257,NaN,NaN,236.503533,236.503533
634,LQN230413610162,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,...,14.365418,6.543361,0.494072,13.962232,15.917473,7.636701,NaN,NaN,236.503533,236.503533


In [28]:
def aggregate_power_measurements(
    agg_cols_energy: list, agg_cols_power: list
) -> pd.DataFrame:
    """
    Aggregate the power data for each measurement by summing the energy values and averaging the power values.
    This is done by grouping the data by ScannerID, ExaminationID, and MeasurementID, and then applying
    the appropriate aggregation functions to the relevant columns. The resulting DataFrame will have one row per measurement with the aggregated energy and power values.

    Returns:
        pd.DataFrame: A DataFrame containing the aggregated energy and power data for each measurement.
    """

    # Group and aggregate
    agg_power_df = power_daily_scannerInfo_df.groupby(
        ["ScannerID_scan", "ExaminationID_exam", "MeasurementID_meas"], as_index=False
    ).agg(
        {
            **{col: "sum" for col in agg_cols_energy},
            **{col: "mean" for col in agg_cols_power},
        }
    )

    # Register the aggregated DataFrame as a DuckDB table if needed
    con.register("agg_power", agg_power_df)

    return agg_power_df


# Define the columns to aggregate for energy, which will be summed
agg_cols_energy = [
    "DailyEcoPowerModeEnergy_KWh_energy",
    "DailyIdleEnergy_KWh_energy",
    "DailyScanEnergy_KWh_energy",
    "DailyTotalEnergy_KWh_energy",
    "TotalEnergy_KWh_exam",
    "TotalEnergy_KWh_meas",
]
# Define the columns to aggregate for power, which will be averaged
agg_cols_power = [
    "TotalActivePower_KW_exam",
    "TotalApparentPower_KVA_exam",
    "TotalReactivePower_KVAR_exam",
    "TotalActivePower_KW_meas",
    "TotalApparentPower_KVA_meas",
    "TotalReactivePower_KVAR_meas",
]

agg_power_df = aggregate_power_measurements(agg_cols_energy, agg_cols_power)
display(agg_power_df)

,ScannerID_scan,ExaminationID_exam,MeasurementID_meas,DailyEcoPowerModeEnergy_KWh_energy,DailyIdleEnergy_KWh_energy,DailyScanEnergy_KWh_energy,DailyTotalEnergy_KWh_energy,TotalEnergy_KWh_exam,TotalEnergy_KWh_meas,TotalActivePower_KW_exam,TotalApparentPower_KVA_exam,TotalReactivePower_KVAR_exam,TotalActivePower_KW_meas,TotalApparentPower_KVA_meas,TotalReactivePower_KVAR_meas
0,52,4764007,1440778853719,24.15323,161.868608,137.312947,323.334785,13.534349,0.089434,18.841875,22.364046,12.024971,14.634663,17.246098,9.124331
1,52,4764007,1440778853826,24.15323,161.868608,137.312947,323.334785,13.534349,0.006509,18.841875,22.364046,12.024971,23.430800,27.281600,13.974380
2,52,4764007,1440778853848,24.15323,161.868608,137.312947,323.334785,13.534349,0.006514,18.841875,22.364046,12.024971,23.448670,27.308740,13.997398
3,52,4764007,1440778853891,24.15323,161.868608,137.312947,323.334785,13.534349,0.019521,18.841875,22.364046,12.024971,23.424660,27.244490,13.912136
4,52,4764007,1440778853913,24.15323,161.868608,137.312947,323.334785,13.534349,0.039116,18.841875,22.364046,12.024971,23.469605,27.319080,13.982379
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
406,55,4952970,1470079439259,0.00000,0.000000,355.758010,355.758010,8.188060,1.128769,14.096415,17.157418,9.566792,15.150501,18.621847,10.654443
407,55,4952970,1470079439374,0.00000,0.000000,355.758010,355.758010,8.188060,0.733657,14.096415,17.157418,9.566792,15.348747,18.592754,10.297279
408,55,4952970,1470079439457,0.00000,0.000000,355.758010,355.758010,8.188060,1.334405,14.096415,17.157418,9.566792,18.762637,22.516678,12.343141
409,55,4952970,1470079439561,0.00000,0.000000,355.758010,355.758010,8.188060,0.972922,14.096415,17.157418,9.566792,13.745892,16.815787,9.524976


In [29]:
def collapse_powermeters(agg_cols_energy: list, agg_cols_power: list) -> pd.DataFrame:
    """
    Collapse the power data for each measurement by dropping duplicates based on ScannerID, ExaminationID, and MeasurementID.
    This is done to ensure that there is only one row per measurement in the resulting DataFrame, which can be useful for analysis and visualization purposes.

    Returns:
        pd.DataFrame: A DataFrame containing the collapsed power data for each measurement with one row per measurement.
    """

    # Drop the PowermeterID_scan since the data will be aggregated over the
    # the different powermeters and we want to have one row per measurement,
    # not per powermeter. Thus, this column would be misleading
    reduced_power_daily_scannerInfo_df = power_daily_scannerInfo_df.drop(
        columns=["PowermeterID_scan"], inplace=False
    )

    # Drop any duplicate rows based on the combination of ScannerID_scan,
    # ExaminationID_exam, and MeasurementID_meas
    reduced_power_daily_scannerInfo_df.drop_duplicates(
        subset=["ScannerID_scan", "ExaminationID_exam", "MeasurementID_meas"],
        inplace=True,
    )

    # Ensure that the columns used for merging are the same in both DataFrames
    merge_columns = ["ScannerID_scan", "ExaminationID_exam", "MeasurementID_meas"]

    if not reduced_power_daily_scannerInfo_df[merge_columns].columns.equals(
        agg_power_df[merge_columns].columns
    ):
        raise ValueError(
            "The columns used for merging do not match between the DataFrames."
        )

    # Merge aggregated results back to deduplicated dataframe
    power_scannerInfo_df = reduced_power_daily_scannerInfo_df.drop(
        columns=agg_cols_energy + agg_cols_power,
    ).merge(
        agg_power_df, on=merge_columns, how="inner", validate="one_to_one", sort=False
    )

    # Optional: Register as DuckDB table if needed
    con.register("power_scannerInfo", power_scannerInfo_df)

    return power_scannerInfo_df


power_scannerInfo_df = collapse_powermeters(agg_cols_energy, agg_cols_power)
display(power_scannerInfo_df)

,ScannerID_scan,SystemType_scan,SiteName_scan,SiteSecondaryName_scan,SiteStreet_scan,SiteCity_scan,CountryShortName_scan,CountryLongName_scan,Serial_scan,CustomName_scan,...,DailyScanEnergy_KWh_energy,DailyTotalEnergy_KWh_energy,TotalEnergy_KWh_exam,TotalEnergy_KWh_meas,TotalActivePower_KW_exam,TotalApparentPower_KVA_exam,TotalReactivePower_KVAR_exam,TotalActivePower_KW_meas,TotalApparentPower_KVA_meas,TotalReactivePower_KVAR_meas
0,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,MR51_AvantoFit,...,137.312947,323.334785,12.047036,0.637343,20.098337,23.887145,12.851608,18.013544,21.365114,11.463432
1,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,MR51_AvantoFit,...,137.312947,323.334785,12.047036,0.004113,20.098337,23.887145,12.851608,14.806590,17.520260,9.366131
2,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,MR51_AvantoFit,...,137.312947,323.334785,12.047036,0.004128,20.098337,23.887145,12.851608,14.859760,17.582910,9.399269
3,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,MR51_AvantoFit,...,137.312947,323.334785,12.047036,0.019292,20.098337,23.887145,12.851608,23.150973,26.514657,12.897089
4,52,AvantoFIT,UKT,MR 4,<NA>,Tübingen,DE,Germany,69667,MR51_AvantoFit,...,137.312947,323.334785,12.047036,0.008857,20.098337,23.887145,12.851608,31.883980,35.576720,15.783372
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
406,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,MR61_Vida,...,355.758010,355.758010,8.188060,1.128769,14.096415,17.157418,9.566792,15.150501,18.621847,10.654443
407,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,MR61_Vida,...,355.758010,355.758010,8.188060,0.733657,14.096415,17.157418,9.566792,15.348747,18.592754,10.297279
408,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,MR61_Vida,...,355.758010,355.758010,8.188060,1.334405,14.096415,17.157418,9.566792,18.762637,22.516678,12.343141
409,55,Vida,UKT,MR 4,<NA>,Tübingen,DE,Germany,75609,MR61_Vida,...,355.758010,355.758010,8.188060,0.972922,14.096415,17.157418,9.566792,13.745892,16.815787,9.524976


## ⚡ Important Note on Energy and Power Differences ⚡

The BETWEEN operator in SQL is inclusive, meaning it includes both the Date_start <br>
and Date_end values. For example, if MeasurementStart_meas is 2024-12-02 16:56:18 <br>
and MeasurementEnd_meas is also 2024-12-02 16:56:18, the date 2024-12-02 16:56:18 <br>
will still be included in the selection. 🕒✅

As a result, the corresponding Energy and Power values can still be aggregated. <br>
This behavior explains the small differences in Energy and Power consumption at <br>
the Measurement level, while the Examination level shows exact matching Energy <br>
and Power consumption. 🔍📊

In [30]:
power_scannerInfo_df[
    [
        "MeasurementID_meas",
        "MeasurementStart_meas",
        "MeasurementEnd_meas",
        "SiemensTotalEnergy_KWh_meas",
        "TotalEnergy_KWh_meas",
        "SiemensTotalEnergy_KWh_exam",
        "TotalEnergy_KWh_exam",
    ]
]

,MeasurementID_meas,MeasurementStart_meas,MeasurementEnd_meas,SiemensTotalEnergy_KWh_meas,TotalEnergy_KWh_meas,SiemensTotalEnergy_KWh_exam,TotalEnergy_KWh_exam
0,1437888325575,2024-12-02 16:54:00,2024-12-02 16:56:05,0.632925,0.637343,12.047036,12.047036
1,1437888325695,2024-12-02 16:56:18,2024-12-02 16:56:18,NaN,0.004113,12.047036,12.047036
2,1437888325716,2024-12-02 16:56:20,2024-12-02 16:56:20,NaN,0.004128,12.047036,12.047036
3,1437888325737,2024-12-02 16:56:21,2024-12-02 16:56:23,0.011521,0.019292,12.047036,12.047036
4,1437888325759,2024-12-02 16:56:24,2024-12-02 16:56:24,NaN,0.008857,12.047036,12.047036
...,...,...,...,...,...,...,...
406,1470079439259,2025-01-20 13:44:20,2025-01-20 13:46:33,1.119902,1.128769,8.188060,8.188060
407,1470079439374,2025-01-20 13:46:33,2025-01-20 13:47:58,0.725728,0.733657,8.188060,8.188060
408,1470079439457,2025-01-20 13:47:59,2025-01-20 13:50:06,1.328965,1.334405,8.188060,8.188060
409,1470079439561,2025-01-20 13:50:12,2025-01-20 13:52:18,0.966636,0.972922,8.188060,8.188060
